In [1]:
# ==========================================================
# STEP 1: DATABASE CONNECTION & VERIFICATION
# ==========================================================

import sqlite3
import pandas as pd
import datetime  # Built into Python: handles dates and timestamps

# Connect to our SQLite school database
conn = sqlite3.connect('school.db')

# Quick test query to verify data exists
test_df = pd.read_sql_query("SELECT COUNT(*) AS total_enrollments FROM enrollments", conn)
print("✅ Database connected successfully!")
print(f"Total enrollment records ready for pipeline: {test_df['total_enrollments'].iloc[0]}")

✅ Database connected successfully!
Total enrollment records ready for pipeline: 15


In [7]:
# ==========================================================
# STEP 2: EXTRACT RAW DATA VIA SQL JOIN
# ==========================================================

query = """ 
       SELECT
       s.student_id,
       s.first_name || ' ' || s.last_name AS full_name, -- || is SQL's way of gluing text together
       s.email,
       s.grade_level,
       c.course_id,
       c.course_name,
       c.subject,
       c.instructor,
       e.semester,
       e.final_grade
       FROM enrollments e
       JOIN students s ON e.student_id = s.student_id
       JOIN courses c ON e.course_id = c.course_id
       ORDER BY s.student_id
"""
# Pull data into a pandas DataFrame

raw_data = pd.read_sql_query(query, conn)
raw_data

,student_id,full_name,email,grade_level,course_id,course_name,subject,instructor,semester,final_grade
0,1,Amina Okafor,amina@school.edu,Year 1,101,Introduction to Python,Computer Science,Dr. Adewale,Fall 2025,88.5
1,1,Amina Okafor,amina@school.edu,Year 1,102,Calculus I,Mathematics,Prof. Schmidt,Fall 2025,72.0
2,2,James Smith,james@school.edu,Year 2,104,Data Structures,Computer Science,Dr. Adewale,Fall 2025,91.0
3,2,James Smith,james@school.edu,Year 2,105,Statistics,Mathematics,Prof. Rossi,Fall 2025,65.5
4,3,Fatima Bello,fatima@school.edu,Year 1,101,Introduction to Python,Computer Science,Dr. Adewale,Fall 2025,95.0
5,3,Fatima Bello,fatima@school.edu,Year 1,103,English Composition,English,Dr. Osei,Fall 2025,82.0
6,4,David Chen,david@school.edu,Year 3,104,Data Structures,Computer Science,Dr. Adewale,Fall 2025,78.5
7,4,David Chen,david@school.edu,Year 3,105,Statistics,Mathematics,Prof. Rossi,Fall 2025,88.0
8,5,Grace Eze,grace@school.edu,Year 2,102,Calculus I,Mathematics,Prof. Schmidt,Fall 2025,55.0
9,5,Grace Eze,grace@school.edu,Year 2,103,English Composition,English,Dr. Osei,Fall 2025,70.0


In [9]:
# ==========================================================
# STEP 3: TRANSFORMATION & ACADEMIC LOGIC
# ==========================================================

# 1. Function to calculate letter grades

def calculate_grade(score):
    if score >= 85:
        return 'A'
    elif score >= 70:
        return 'B'
    elif score >= 55:
        return 'C'
    elif score >= 50:
        return 'D'
    else:
        return 'F'

# 2. Function to determine student standing

def evaluate_status(score):
    if score >= 90:
        return 'Dean\s Honor Roll'
    elif score < 50:
        return 'Academic Warning'
    else:
        return 'Satisfactory'

# Apply the functions across our dataset
raw_data['letter_grade'] = raw_data['final_grade'].apply(calculate_grade)
raw_data['academic_status'] = raw_data['final_grade'].apply(evaluate_status)

print("---ENRICHED DATASET WITH STATUS FLAGS---")
raw_data[['full_name', 'course_name', 'final_grade', 'letter_grade', 'academic_status']]

---ENRICHED DATASET WITH STATUS FLAGS---


,full_name,course_name,final_grade,letter_grade,academic_status
0,Amina Okafor,Introduction to Python,88.5,A,Satisfactory
1,Amina Okafor,Calculus I,72.0,B,Satisfactory
2,James Smith,Data Structures,91.0,A,Dean\s Honor Roll
3,James Smith,Statistics,65.5,C,Satisfactory
4,Fatima Bello,Introduction to Python,95.0,A,Dean\s Honor Roll
5,Fatima Bello,English Composition,82.0,B,Satisfactory
6,David Chen,Data Structures,78.5,B,Satisfactory
7,David Chen,Statistics,88.0,A,Satisfactory
8,Grace Eze,Calculus I,55.0,C,Satisfactory
9,Grace Eze,English Composition,70.0,B,Satisfactory


In [10]:
# ==========================================================
# STEP 4: AUTOMATED MULTI-REPORT EXPORT PIPELINE
# ==========================================================

# Get today's date in YYYY-MM-DD format

today_str = today_str = datetime.date.today().strftime('%Y-%m-%d')
print(f"Report Generation Date: {today_str}\n")



Report Generation Date: 2026-10-06



In [26]:
# ---------------------------------------------------------
# REPORT 1: EXECUTIVE DEPARTMENT SUMMARY (For the Dean)
# ---------------------------------------------------------

dept_summary = raw_data.groupby('subject').agg(
    total_enrollments=('final_grade', 'count'),
    average_score=('final_grade', 'mean'),
    highest_score=('final_grade', 'max'),
    lowest_score=('final_grade', 'min')
    
).round(1)
dept_summary_filename = f"Executive_Summary_{today_str}.csv"
dept_summary.to_csv(dept_summary_filename)


# ---------------------------------------------------------
# REPORT 2: ACTION LIST - ACADEMIC WARNINGS (For Advisors)
# ---------------------------------------------------------

at_risk_list = raw_data[raw_data['academic_status'] == 'Academic Warning'][
    ['student_id', 'full_name', 'email', 'course_name', 'instructor', 'final_grade', 'letter_grade']
]
at_risk_filename = f"Academic_Warning_List_{today_str}.csv"
at_risk_list.to_csv(at_risk_filename, index=False)




# ---------------------------------------------------------
# REPORT 3: DEAN'S HONOR ROLL (For Recognition/Awards)
# ---------------------------------------------------------

honor_roll_list = raw_data[raw_data['academic_status'] == "Dean's Honor Roll"][
    ['student_id', 'full_name', 'course_name', 'instructor', 'final_grade']
].sort_values('final_grade', ascending=False)



honor_roll_list = raw_data[raw_data['academic_status'] == "Dean's Honor Roll"][
    ['student_id', 'full_name', 'course_name', 'instructor', 'final_grade']
].sort_values('final_grade', ascending=False).reset_index(drop=True)

honor_roll_list


,student_id,full_name,course_name,instructor,final_grade
0,7,Sarah Williams,Data Structures,Dr. Adewale,96.0
1,3,Fatima Bello,Introduction to Python,Dr. Adewale,95.0
2,10,Kwame Asante,Statistics,Prof. Rossi,91.5
3,2,James Smith,Data Structures,Dr. Adewale,91.0


In [27]:
# ==========================================================
# STEP 5: THE MASTER AUTOMATION FUNCTION
# ==========================================================

def run_weekly_lms_pipeline():
    """
    Complete automated ETL pipeline:
    Extracts from SQLite, cleans data, calculates academic standing,
    and exports timestamped CSV reports.
    """
    print("=" * 60)
    print(f"⚙️ EXECUTING LMS DATA PIPELINE: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)
    
    # 1. Extract
    sql_query = """
        SELECT 
            s.student_id,
            s.first_name || ' ' || s.last_name AS full_name,
            s.email,
            s.grade_level,
            c.course_name,
            c.subject,
            c.instructor,
            e.final_grade
        FROM enrollments e
        JOIN students s ON e.student_id = s.student_id
        JOIN courses c ON e.course_id = c.course_id;
    """
    df_pipeline = pd.read_sql_query(sql_query, conn)
    
    # 2. Transform
    df_pipeline['letter_grade'] = df_pipeline['final_grade'].apply(calculate_grade)
    df_pipeline['academic_status'] = df_pipeline['final_grade'].apply(evaluate_status)
    
    # 3. Export
    date_stamp = datetime.date.today().strftime('%Y-%m-%d')
    
    summary_file = f"Auto_Department_Summary_{date_stamp}.csv"
    warnings_file = f"Auto_Academic_Warnings_{date_stamp}.csv"
    
    # Department summary export
    summary = df_pipeline.groupby('subject')['final_grade'].agg(['count', 'mean']).round(1)
    summary.to_csv(summary_file)
    
    # Warnings export
    warnings = df_pipeline[df_pipeline['academic_status'] == 'Academic Warning']
    warnings.to_csv(warnings_file, index=False)
    
    print("✅ PIPELINE EXECUTION SUCCESSFUL!")
    print(f"📁 Generated: {summary_file}")
    print(f"📁 Generated: {warnings_file}")
    print(f"👥 Processed {len(df_pipeline)} enrollment records.")
    print("=" * 60)

# TEST RUN: Run the entire automated pipeline with one line!
run_weekly_lms_pipeline()
    

⚙️ EXECUTING LMS DATA PIPELINE: 2026-10-06 22:02:45
✅ PIPELINE EXECUTION SUCCESSFUL!
📁 Generated: Auto_Department_Summary_2026-10-06.csv
📁 Generated: Auto_Academic_Warnings_2026-10-06.csv
👥 Processed 15 enrollment records.
